In [4]:
# Notebooks live in Notebooks/, but every data path is relative to the repo root.
import os, sys
while not os.path.isdir("data") and os.getcwd() != "/":
    os.chdir("..")
if os.getcwd() not in sys.path:
    sys.path.insert(0, os.getcwd())
print("cwd:", os.getcwd())

cwd: /home/lotanamit/WhatDoLLMsWant


In [6]:
import os
import pandas as pd
import numpy as np

datadir = 'data'
exp_name = 'laptops_robustness'
job_id = '68337596'
path = os.path.join(datadir, exp_name, job_id, 'scores.csv')
df = pd.read_csv(path)
df.head()

,template_idx,score_a,score_b,a_brand,a_screen,a_ram,b_brand,b_screen,b_ram
0,0,-21.671875,6.039062,ASUS,13-inch,4GB,ASUS,13-inch,8GB
1,0,-17.593750,6.039062,ASUS,13-inch,4GB,ASUS,13-inch,16GB
2,0,-21.734375,6.039062,ASUS,13-inch,4GB,ASUS,14-inch,4GB
3,0,-25.250000,6.039062,ASUS,13-inch,4GB,ASUS,14-inch,8GB
4,0,-23.984375,6.039062,ASUS,13-inch,4GB,ASUS,14-inch,16GB


In [13]:
from src.pref_models import fit_feature_based_bradley_terry

bt = fit_feature_based_bradley_terry(df)

Feature p-values:
  brand_Apple: p-value = 0.0000
  brand_Dell: p-value = 0.0006
  brand_HP: p-value = 0.0046
  brand_Lenovo: p-value = 0.0310
  screen_14-inch: p-value = 0.0000
  screen_16-inch: p-value = 0.6567
  ram_4GB: p-value = 0.0000
  ram_8GB: p-value = 0.0000
Positional bias (gamma): -8.9796, p-value = 0.0000
Model R-squared: 0.8170, F-statistic p-value: 0.0000


In [15]:
bt['feature_scores']

{'brand_Apple': np.float64(-1.278200314368716),
 'brand_Dell': np.float64(-0.5054182189187797),
 'brand_HP': np.float64(-0.4194378360701032),
 'brand_Lenovo': np.float64(-0.3196018774715419),
 'screen_14-inch': np.float64(8.546114293981486),
 'screen_16-inch': np.float64(-0.05100584129050435),
 'ram_4GB': np.float64(-21.852601476598664),
 'ram_8GB': np.float64(-9.019257057472508)}

In [ ]:
import warnings

import statsmodels.api as sm
from scipy.special import expit


class FeatureBT:
    """Feature-based Bradley-Terry with positional bias (OLS on score differences).

    weights_ holds every level of every feature, centered to mean 0 within each
    feature: positive = above average for that feature, negative = below.
    gamma_ (positional bias) is kept separate and never enters utilities.
    """

    def __init__(self, r2_warn=0.7, pvalue_warn=0.05):
        self.r2_warn = r2_warn
        self.pvalue_warn = pvalue_warn

    def fit(self, df, a_prefix="a_", b_prefix="b_",
            score_a_col="score_a", score_b_col="score_b"):
        n = len(df)
        Y = (df[score_a_col] - df[score_b_col]).values

        cols_a = [c for c in df.columns if c.startswith(a_prefix)]
        cols_b = [c.replace(a_prefix, b_prefix) for c in cols_a]
        self.features_ = [c[len(a_prefix):] for c in cols_a]

        df_a = df[cols_a].rename(columns=dict(zip(cols_a, self.features_)))
        df_b = df[cols_b].rename(columns=dict(zip(cols_b, self.features_)))
        combined = pd.concat([df_a, df_b], axis=0)

        # drop_first=True: one reference level per feature is pinned to 0 so OLS
        # is full-rank; we add it back and re-center below.
        dummies = pd.get_dummies(combined, drop_first=True, dtype=float)
        X = sm.add_constant(dummies.iloc[:n].values - dummies.iloc[n:].values)

        self.result_ = sm.OLS(Y, X).fit()
        self.gamma_ = self.result_.params[0]
        self.r2_ = self.result_.rsquared

        dummy_names = dummies.columns.tolist()
        raw = dict(zip(dummy_names, self.result_.params[1:]))
        self.pvalues_ = dict(zip(dummy_names, self.result_.pvalues[1:]))

        # All levels per feature, reference level = 0, then center to mean 0
        self.weights_ = {}
        for feat in self.features_:
            w = {lvl: raw.get(f"{feat}_{lvl}", 0.0) for lvl in combined[feat].unique()}
            mean = np.mean(list(w.values()))
            self.weights_[feat] = {lvl: v - mean for lvl, v in w.items()}

        if self.r2_ < self.r2_warn:
            warnings.warn(f"Low fit: R^2 = {self.r2_:.3f} < {self.r2_warn}")
        weak = {k: p for k, p in self.pvalues_.items() if p > self.pvalue_warn}
        if weak:
            weak_str = ", ".join(f"{k} (p={p:.3f})" for k, p in weak.items())
            warnings.warn(f"Weak levels (vs reference, p > {self.pvalue_warn}): {weak_str}")
        return self

    def utility(self, item: dict) -> float:
        # item = {"brand": "Apple", "screen": "14-inch", "ram": "16GB"}
        # KeyError on unknown feature/level is intentional (catches typos)
        return sum(self.weights_[f][item[f]] for f in self.features_)

    def predict(self, item_a: dict, item_b: dict) -> float:
        """P(A beats B), position-free (gamma excluded)."""
        return expit(self.utility(item_a) - self.utility(item_b))

In [ ]:
model = FeatureBT().fit(df)

print("R^2:", round(model.r2_, 4), "| gamma:", round(model.gamma_, 4))
for feat, w in model.weights_.items():
    print(feat, {lvl: round(v, 3) for lvl, v in w.items()})

best = {"brand": "Apple", "screen": "14-inch", "ram": "16GB"}
worst = {"brand": "Apple", "screen": "13-inch", "ram": "4GB"}
print("utility(best):", round(model.utility(best), 3))
print("utility(worst):", round(model.utility(worst), 3))
print("P(best beats worst):", round(model.predict(best, worst), 4))

## Two adherence metrics — notation

**Model.** For a pair (a, b) the fit is

$$Y = \text{score}_a - \text{score}_b = \gamma + u(a) - u(b) + \varepsilon,
\qquad u(x) = \sum_{f} w_f(x_f), \qquad \sum_{\ell} w_f(\ell) = 0 \ \text{per feature}.$$

$\gamma$ is the positional bias; it is not part of any laptop.

**Setup.** The contract names feature $f$ and asks for level $\ell^{*}$ (14-inch, 8GB).
The model's own favourite level that violates the contract is the rival $\ell'$ (16-inch, 16GB).
The **preference gap** on the named feature in run $r$ is

$$g_r = w_f^{(r)}(\ell^{*}) - w_f^{(r)}(\ell'), \qquad r \in \{0 = \text{no contract},\ 1 = \text{contract}\}.$$

$g_0 < 0$ means the model starts out preferring the rival — the conflict case.

**Metric 1 — Compliance $C$ (behaviour).** On the conflict set $S$ = ceteris-paribus pairs
($a,b$ identical except feature $f$, one side $\ell^{*}$, the other $\ell'$):

$$C = \frac{1}{|S|} \sum_{i \in S} \mathbf{1}\!\left[\, \text{the } \ell^{*} \text{ side wins} \,\right],
\qquad \text{winner decided by the de-biased margin } Y_i - \hat{\gamma}.$$

$C = 50\%$: contract changed nothing; $C = 100\%$: contract always decides. Saturates near the ends.

**Metric 2 — Cancellation $\kappa$ (preference).** How much of its own gap the contract cancelled:

$$\kappa = 1 - \frac{g_1}{g_0}.$$

$\kappa = 0$: contract did nothing; $\kappa = 1$: model now exactly indifferent;
$\kappa > 1$: the sign flipped — the contract overrode the preference, with margin $\kappa - 1$.
The weight scale cancels in the ratio, so $\kappa$ is comparable across models.
Undefined when $g_0 \approx 0$ (no own preference to cancel).

In [ ]:
# Test both metrics on qwen-32B: compliance C (behaviour) vs cancellation kappa (preference).
import json

BASES = ["data/laptops_robustness", "data/laptops_robustness_gemma"]


def load_run(fam, size, con_id):
    for base in BASES:
        for run in sorted(os.listdir(base)):
            cp = os.path.join(base, run, "config.json")
            if not os.path.isfile(cp):
                continue
            c = json.load(open(cp))
            if (c["model_family"], str(c["model_size"]), c["constraints_id"]) == (fam, size, con_id):
                return pd.read_csv(os.path.join(base, run, "scores.csv"))
    raise KeyError((fam, size, con_id))


ASKED  = {"screen": "14-inch", "ram": "8GB"}    # the level the contract asks for
RIVAL  = {"screen": "16-inch", "ram": "16GB"}   # the model's own favourite -> conflict
RUN_OF = {"screen": "screen=14-inch", "ram": "ram=8GB"}


def compliance(d, fit, feat):
    """C: % of ceteris-paribus conflict pairs where the asked-for level wins,
    winner decided by the de-biased margin Y - gamma."""
    same = np.ones(len(d), bool)
    for f in fit.features_:
        if f != feat:
            same &= (d[f"a_{f}"] == d[f"b_{f}"]).values
    pair = (((d[f"a_{feat}"] == ASKED[feat]) & (d[f"b_{feat}"] == RIVAL[feat]))
            | ((d[f"a_{feat}"] == RIVAL[feat]) & (d[f"b_{feat}"] == ASKED[feat]))).values
    m = same & pair
    asked_is_a = (d[f"a_{feat}"] == ASKED[feat]).values[m]
    margin = (d.score_a - d.score_b).values[m] - fit.gamma_
    asked_margin = np.where(asked_is_a, margin, -margin)
    return 100 * (asked_margin > 0).mean(), int(m.sum())


MODELS = [("qwen", "32")]  # extend here for more models

rows = []
for fam, size in MODELS:
    runs = {con: load_run(fam, size, con) for con in ["none"] + list(RUN_OF.values())}
    fits = {con: FeatureBT().fit(d) for con, d in runs.items()}
    for feat in ["screen", "ram"]:
        con = RUN_OF[feat]
        w0, w1 = fits["none"].weights_[feat], fits[con].weights_[feat]
        g0 = w0[ASKED[feat]] - w0[RIVAL[feat]]
        g1 = w1[ASKED[feat]] - w1[RIVAL[feat]]
        C1, n = compliance(runs[con], fits[con], feat)
        C0, _ = compliance(runs["none"], fits["none"], feat)
        rows.append(dict(model=f"{fam}-{size}B", feature=feat,
                         pair=f"{ASKED[feat]} vs {RIVAL[feat]}", n_pairs=n,
                         C_before=round(C0, 1), C_after=round(C1, 1),
                         g0=round(g0, 2), g1=round(g1, 2),
                         kappa=round(1 - g1 / g0, 2), flipped=bool(g1 > 0)))

pd.DataFrame(rows)